# SPM & Hue Angle Analysis with Land Masking 
Analysis of high-sediment areas using SPM and Hue Angle data and land masking using a coastline shapefile.

In [ ]:
import rasterio
from rasterio import features, mask
import numpy as np
import geopandas as gpd
from shapely.geometry import shape, MultiPolygon
import os
from skimage.filters import threshold_otsu, threshold_li, threshold_yen, gaussian
from skimage import feature
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.ndimage import distance_transform_edt

base_path = r'../data/Area2'
coast_path = r'../ROI/iho.shp'
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [ ]:


def extract_polygons(data, threshold, transform, crs, output_path, condition='>', min_area=5000):
    if condition == '>':
        m = (data > threshold) & (~np.isnan(data))
    else:
        m = (data < threshold) & (~np.isnan(data))
    
    results = (
        {'properties': {'raster_val': v}, 'geometry': s}
        for i, (s, v) 
        in enumerate(features.shapes(m.astype(np.int16), mask=m, transform=transform))
    )
    
    geoms = list(results)
    if not geoms: return 0
    
    gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
    gdf['area_m2'] = gdf.geometry.area
    gdf = gdf[gdf['area_m2'] > min_area]
    
    if not gdf.empty:
        gdf.to_file(output_path, driver='GPKG')
        return len(gdf)
    
    return 0


def apply_land_mask(src_path, coast_gdf):
    with rasterio.open(src_path) as src:
        try:
            out_img, _ = mask.mask(src, coast_gdf.to_crs(src.crs).geometry, invert=False)
            data = out_img[0]
            data[data == 0] = np.nan
            return data, src.transform, src.crs
        except:
            return src.read(1), src.transform, src.crs




##  Single Month Analysis 
Analyzing SPM and Hue Angle with Land Masking.

In [ ]:

month = 'Jan'
coast_gdf = gpd.read_file(coast_path)

spm_path = os.path.join(base_path, month, 'monthly_median_spm_masked.tif')
hue_path = os.path.join(base_path, month, 'monthly_median_hue_masked.tif')

# -------------------------
# 1.1 SPM Processing
# -------------------------
spm_data, transform, crs = apply_land_mask(spm_path, coast_gdf)


spm_thresh = np.nanpercentile(spm_data, 90)
print(f'SPM Threshold: {spm_thresh:.2f}')
print(np.round(np.nanpercentile(spm_data, [50, 75, 90, 95, 99]),2))


### Extracting polygon SPM

In [ ]:

extract_polygons(
    spm_data, spm_thresh, transform, crs,
    os.path.join(base_path, month, f'high_spm_{month}_masked.shp'),
    condition='>'
)


In [ ]:

# -------------------------
# 1.2 Hue Processing (STATIC)
# -------------------------
hue_data, _, _ = apply_land_mask(hue_path, coast_gdf)

HUE_THRESH = 120
print(f'Static Hue Threshold: {HUE_THRESH}')


### Extracting polygon hue

In [ ]:

extract_polygons(
    hue_data, HUE_THRESH, transform, crs,
    os.path.join(base_path, month, f'low_hue_{month}_masked.shp'),
    condition='<'
)


In [ ]:

# -------------------------
# 1.3 Visualization
# -------------------------
from scipy.ndimage import median_filter, gaussian_filter
#from matplotlib.colors import LogNorm

# Smooth
spm_smooth = median_filter(spm_data, size=3)

# Scaling
#vmax = np.nanpercentile(spm_data, 98)

# Plot

# --- PLOT ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# SPM plot
im1 = ax1.imshow(spm_smooth, cmap='turbo', vmin=5, vmax=20)
ax1.set_title(f'SPM (thr={spm_thresh:.2f})')

# Hue plot
im2 = ax2.imshow(hue_data, cmap='hsv', vmin=40, vmax=220)
ax2.set_title(f'Hue (thr<{HUE_THRESH})')

# Remove axes
for ax in [ax1, ax2]:
    ax.axis('off')

# --- COLORBARS ---
cbar1 = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label("SPM")

cbar2 = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label("Hue")

plt.tight_layout()
plt.show()

## Defining SPM and hue layers for colormap


In [ ]:
src = rasterio.open(spm_path)
print(src.crs)
print(src.transform)
print(abs(src.transform.a))

### Handling offshore bias

In [ ]:


from scipy.ndimage import (
    distance_transform_edt,
    binary_opening,
    binary_closing,
    label
)

from matplotlib.colors import ListedColormap

# =========================================================
# 1. VALID PIXELS
# =========================================================

valid_mask = (
    np.isfinite(spm_data) &
    np.isfinite(hue_data)
)

# =========================================================
# 2. INITIAL LAND MASK
# =========================================================
# Start from invalid pixels
# (temporary approximation of land)

land_mask = ~valid_mask

# =========================================================
# 3. CLEAN LAND MASK
# =========================================================
# Remove offshore speckles / tiny gaps

land_mask = binary_closing(
    land_mask,
    structure=np.ones((7, 7))
)

land_mask = binary_opening(
    land_mask,
    structure=np.ones((7, 7))
)

# =========================================================
# 4. KEEP ONLY LARGEST LAND COMPONENT
# =========================================================
# Removes offshore noise blobs

labeled, num = label(land_mask)

sizes = np.bincount(labeled.ravel())

# Ignore background index 0
largest_component = sizes[1:].argmax() + 1

land_mask = labeled == largest_component

# =========================================================
# 5. DISTANCE TO LAND
# =========================================================
# EPSG:32644 → pixel size already in meters

pixel_size = abs(transform.a)

distance_to_land = (
    distance_transform_edt(~land_mask)
    * pixel_size
)

# =========================================================
# 6. COASTAL / OFFSHORE ZONES
# =========================================================

MAX_DIST = 15000  # 15 km

coastal_zone = distance_to_land <= MAX_DIST
offshore_zone = distance_to_land > MAX_DIST

# =========================================================
# 7. THRESHOLD MASKS
# =========================================================

spm_mask = spm_data > spm_thresh
hue_mask = hue_data < HUE_THRESH

# =========================================================
# 8. COASTAL-RESTRICTED MASKS
# =========================================================

coastal_spm = (
    valid_mask &
    spm_mask &
    coastal_zone
)

coastal_hue = (
    valid_mask &
    hue_mask &
    coastal_zone
)

plume_mask = (
    valid_mask &
    spm_mask &
    hue_mask &
    coastal_zone
)

offshore_bias = (
    valid_mask &
    spm_mask &
    offshore_zone
)


### Colour classification

In [ ]:

classification = np.zeros_like(
    spm_data,
    dtype=np.uint8
)

# 0 = no data / land

classification[valid_mask] = 1                 # water
classification[coastal_spm & ~hue_mask] = 2    # high SPM
classification[coastal_hue & ~spm_mask] = 3    # low hue
classification[plume_mask] = 4                 # plume
classification[offshore_bias] = 5              # offshore artifact

# =========================================================
# 10. COLORMAP
# =========================================================

cmap = ListedColormap([
    'black',      # 0 no data / land
    'lightblue',  # 1 water
    'orange',     # 2 high SPM
    'purple',     # 3 low hue
    'red',        # 4 plume
    'gray'        # 5 offshore bias
])

In [ ]:

# =========================================================
# 11. DEBUG PLOTS (VERY IMPORTANT)
# =========================================================

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].imshow(land_mask)
ax[0].set_title("Land Mask")

im = ax[1].imshow(distance_to_land)
ax[1].set_title("Distance to Land")
plt.colorbar(im, ax=ax[1])

ax[2].imshow(coastal_zone)
ax[2].set_title("Coastal Zone")

plt.tight_layout()
plt.show()

### Reprojecting CRS

In [ ]:

from rasterio.warp import reproject, Resampling, transform_bounds

# ----------------------------
# 1. Open raster
# ----------------------------
src = rasterio.open(spm_path)

spm = src.read(1)
classification = np.array(classification)

if classification.shape != spm.shape:
    raise ValueError(f"Shape mismatch: {classification.shape} vs {spm.shape}")

# ----------------------------
# 2. FORCE CRS WITHOUT EPSG LOOKUP
# ----------------------------
# Use PROJ string instead of "EPSG:4326"
dst_crs = "+proj=longlat +datum=WGS84 +no_defs"

# ----------------------------
# 3. Compute bounds transform WITHOUT calculate_default_transform
# ----------------------------
dst_bounds = transform_bounds(
    src.crs,
    dst_crs,
    *src.bounds,
    densify_pts=21
)

minx, miny, maxx, maxy = dst_bounds

# define output grid manually
width = src.width
height = src.height

dst_transform = rasterio.transform.from_bounds(
    minx, miny, maxx, maxy,
    width, height
)

# ----------------------------
# 4. Prepare output arrays
# ----------------------------
spm_wgs84 = np.empty((height, width), dtype=spm.dtype)
classification_wgs84 = np.empty((height, width), dtype=classification.dtype)

# ----------------------------
# 5. Reproject SPM (continuous)
# ----------------------------
reproject(
    source=spm,
    destination=spm_wgs84,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.bilinear
)

# ----------------------------
# 6. Reproject classification (categorical)
# ----------------------------
reproject(
    source=classification,
    destination=classification_wgs84,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.nearest
)


### Plotting classified colormap <3

In [ ]:


cmap = ListedColormap([
    'black', 'lightblue', 'orange', 'purple', 'red', 'gray'
])

plt.figure(figsize=(10,6))

plt.imshow(classification_wgs84, cmap=cmap, vmin=0, vmax=5, extent=[minx, maxx, miny, maxy], origin='upper')

cbar = plt.colorbar(ticks=[0,1,2,3,4,5])
cbar.ax.set_yticklabels([
    'Land/no data',
    'SPM <P90',
    'SPM P90',
    'Hue < 120$^\circ$',
    'Hue < 120$^\circ$ & SPM P90',
    'Offshore bias'
])

plt.xlabel("Longitude ($^\circ$E)")
plt.ylabel("Latitude ($^\circ$N)")
plt.title("January")
plt.show()